In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("synthetic_customer_churn_data.csv")
df

,account_id,product_id,txn_amt,date_id,txn_flow
0,1675009,14,395.39,6172,debit
1,1006123,28,409.89,6181,debit
2,1936846,74,56.50,6450,debit
3,1542862,104,43.09,6370,debit
4,1648106,97,849.56,6154,debit
...,...,...,...,...,...
9999995,1102857,21,281.49,6617,debit
9999996,1899285,86,799.01,6195,debit
9999997,1807224,65,121.67,6179,debit
9999998,1046974,96,143.07,6055,debit


### Calculate overall facts
- Exclude latest 30 days data
- Groupby account_id, txn_flow get following overall facts:
    - overall_cnt: total count of account_id,
    - overall_amt: sum of txn_amt,
    - recency : first get max of date_id and then subtract it from max_date_id
- Pivot all facts based on index as account_id, column as txn_flow and values as [overal_cnt, overall_amt, recency] to get other facts(overall_cnt_credit, overall_cnt_debit,	overall_cnt_services, overall_amt_credit, overall_amt_debit, overall_amt_services, recency_credit, recency_debit, recency_services)
<!-- - recency : min of recency_credit,recency_debit,recency_services -->
- account_age : get min of date_id and then subtract it from max_date_id then we get account_age in days

In [3]:
max_date_id = df["date_id"].max() - 30
max_date_id

6699

In [7]:
overall_facts = df.query(f"date_id <= {max_date_id}").groupby(by=["account_id","txn_flow"]).agg(overall_cnt = ("account_id","count"),overall_amt=("txn_amt","sum"),recency=("date_id","max")).reset_index()
overall_facts["recency"] = max_date_id - overall_facts["recency"]
overall_facts


,account_id,txn_flow,overall_cnt,overall_amt,recency
0,1000000,credit,1,128.90,160
1,1000000,debit,7,1990.16,243
2,1000001,credit,4,2094.69,486
3,1000001,debit,3,1181.73,338
4,1000001,services,1,870.82,403
...,...,...,...,...,...
2788456,1999998,debit,3,1288.61,121
2788457,1999998,services,3,1734.86,36
2788458,1999999,credit,2,702.97,397
2788459,1999999,debit,6,3214.19,128


In [8]:
overall_facts["recency"].min(),overall_facts["recency"].max()

(0, 699)

In [9]:
overall_facts = overall_facts.pivot(index="account_id",columns="txn_flow",values=["overall_cnt","overall_amt","recency"]).reset_index()
overall_facts.columns = ["_".join(filter(None, map(str,col))) for col in overall_facts.columns]
overall_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services
0,1000000,1.0,7.0,NaN,128.90,1990.16,NaN,160.0,243.0,NaN
1,1000001,4.0,3.0,1.0,2094.69,1181.73,870.82,486.0,338.0,403.0
2,1000002,3.0,3.0,2.0,2184.48,1357.24,1282.57,147.0,76.0,310.0
3,1000003,2.0,4.0,3.0,965.01,1861.11,1387.66,209.0,6.0,23.0
4,1000004,2.0,4.0,NaN,1412.47,1016.07,NaN,86.0,88.0,NaN
...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7.0,1.0,2.0,3202.84,631.76,1350.68,44.0,72.0,371.0
999930,1999996,4.0,5.0,2.0,1748.08,2244.33,978.72,29.0,358.0,219.0
999931,1999997,1.0,6.0,1.0,933.60,2117.65,778.14,410.0,82.0,172.0
999932,1999998,3.0,3.0,3.0,1124.25,1288.61,1734.86,211.0,121.0,36.0


In [10]:
# overall_facts["recency"] = overall_facts[["recency_credit","recency_debit","recency_services"]].min(axis=1)
overall_facts = overall_facts.fillna({"overall_cnt_credit":0,"overall_cnt_debit":0,"overall_cnt_services":0,"overall_amt_credit":0,"overall_amt_debit":0,"overall_amt_services":0,"recency_credit":-1,"recency_debit":-1,"recency_services":-1})
overall_facts
overall_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services
0,1000000,1.0,7.0,0.0,128.90,1990.16,0.00,160.0,243.0,-1.0
1,1000001,4.0,3.0,1.0,2094.69,1181.73,870.82,486.0,338.0,403.0
2,1000002,3.0,3.0,2.0,2184.48,1357.24,1282.57,147.0,76.0,310.0
3,1000003,2.0,4.0,3.0,965.01,1861.11,1387.66,209.0,6.0,23.0
4,1000004,2.0,4.0,0.0,1412.47,1016.07,0.00,86.0,88.0,-1.0
...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7.0,1.0,2.0,3202.84,631.76,1350.68,44.0,72.0,371.0
999930,1999996,4.0,5.0,2.0,1748.08,2244.33,978.72,29.0,358.0,219.0
999931,1999997,1.0,6.0,1.0,933.60,2117.65,778.14,410.0,82.0,172.0
999932,1999998,3.0,3.0,3.0,1124.25,1288.61,1734.86,211.0,121.0,36.0


In [11]:
account_age = df.query(f"date_id <= {max_date_id}").groupby("account_id").agg(account_age=("date_id","min"))
account_age["account_age"] = max_date_id - account_age["account_age"]

overall_facts = overall_facts.merge(account_age,on="account_id",how="inner")
overall_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services,account_age
0,1000000,1.0,7.0,0.0,128.90,1990.16,0.00,160.0,243.0,-1.0,648
1,1000001,4.0,3.0,1.0,2094.69,1181.73,870.82,486.0,338.0,403.0,688
2,1000002,3.0,3.0,2.0,2184.48,1357.24,1282.57,147.0,76.0,310.0,613
3,1000003,2.0,4.0,3.0,965.01,1861.11,1387.66,209.0,6.0,23.0,493
4,1000004,2.0,4.0,0.0,1412.47,1016.07,0.00,86.0,88.0,-1.0,486
...,...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7.0,1.0,2.0,3202.84,631.76,1350.68,44.0,72.0,371.0,609
999930,1999996,4.0,5.0,2.0,1748.08,2244.33,978.72,29.0,358.0,219.0,630
999931,1999997,1.0,6.0,1.0,933.60,2117.65,778.14,410.0,82.0,172.0,578
999932,1999998,3.0,3.0,3.0,1124.25,1288.61,1734.86,211.0,121.0,36.0,454


In [12]:
overall_facts = overall_facts.astype({'overall_cnt_credit': 'int', 'overall_cnt_debit': 'int','overall_cnt_services':'int','recency_credit':'int','recency_debit':'int','recency_services':'int'})
overall_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services,account_age
0,1000000,1,7,0,128.90,1990.16,0.00,160,243,-1,648
1,1000001,4,3,1,2094.69,1181.73,870.82,486,338,403,688
2,1000002,3,3,2,2184.48,1357.24,1282.57,147,76,310,613
3,1000003,2,4,3,965.01,1861.11,1387.66,209,6,23,493
4,1000004,2,4,0,1412.47,1016.07,0.00,86,88,-1,486
...,...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7,1,2,3202.84,631.76,1350.68,44,72,371,609
999930,1999996,4,5,2,1748.08,2244.33,978.72,29,358,219,630
999931,1999997,1,6,1,933.60,2117.65,778.14,410,82,172,578
999932,1999998,3,3,3,1124.25,1288.61,1734.86,211,121,36,454


### Intermediate result from Identifying Churn Patterns algorithm
- Exclude latest 30 days data
- txn_cnt_3_credit: txn count of credit txn_flow for segment 3
- txn_cnt_3_debit: txn count of debit txn_flow for segment 3
- txn_cnt_3_services: txn count of services txn_flow for segment 3
- txn_cnt_2_credit: txn count of credit txn_flow for segment 2
- txn_cnt_2_debit: txn count of debit txn_flow for segment 2
- txn_cnt_2_services: txn count of services txn_flow for segment 2
- txn_amt_3_credit: sum of txn_amt of credit txn_flow for segment 3
- txn_amt_3_debit: sum of txn_amt of debit txn_flow for segment 3
- txn_amt_3_services: sum of txn_amt of services txn_flow for segment 3
- txn_amt_2_credit: sum of txn_amt of credit txn_flow for segment 2
- txn_amt_2_debit: sum of txn_amt of debit txn_flow for segment 2
- txn_amt_2_services: sum of txn_amt of services txn_flow for segment 2

In [13]:
additional_facts = pd.read_parquet("result/facts/additional_fact.parquet").astype({'txn_cnt_3_credit':'int','txn_cnt_3_debit':'int','txn_cnt_3_services':'int','txn_cnt_2_credit':'int','txn_cnt_2_debit':'int','txn_cnt_2_services':'int'})
additional_facts

,account_id,txn_cnt_3_credit,txn_cnt_3_debit,txn_cnt_3_services,txn_cnt_2_credit,txn_cnt_2_debit,txn_cnt_2_services,txn_amt_3_credit,txn_amt_3_debit,txn_amt_3_services,txn_amt_2_credit,txn_amt_2_debit,txn_amt_2_services
0,1000000,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00
1,1000003,0,0,0,0,1,1,0.00,0.0,0.00,0.00,233.94,843.25
2,1000005,0,0,0,0,1,0,0.00,0.0,0.00,0.00,128.85,0.00
3,1000006,0,0,0,1,0,0,0.00,0.0,0.00,72.60,0.00,0.00
4,1000007,0,1,0,0,0,0,0.00,969.2,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
711890,1999994,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00
711891,1999995,1,0,0,0,0,0,74.04,0.0,0.00,0.00,0.00,0.00
711892,1999996,0,0,0,1,0,0,0.00,0.0,0.00,625.19,0.00,0.00
711893,1999997,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00


In [14]:
txn_flow_facts = pd.read_parquet("result/facts/txn_flow_fact.parquet").drop(columns=["current","seq"])
txn_flow_facts

,account_id,second_last,last,is_churn
0,1000000,0,0,0
1,1000003,0,6,1
2,1000005,0,2,1
3,1000006,0,1,1
4,1000007,2,0,0
...,...,...,...,...
711890,1999994,0,0,0
711891,1999995,1,0,1
711892,1999996,0,1,1
711893,1999997,0,0,0


### Facts Descriptions(result from churn pattern algorithm)
1. second_last : txn_flow_id for 3 segment
2. last: txn_flow_id for 2 segment

**Combining all results: overall facts + additional facts + txn_flow facts**

In [15]:
pd.set_option('display.max_columns', None)

all_facts = overall_facts.merge(additional_facts,on="account_id",how="left").merge(txn_flow_facts,on="account_id",how="left")
all_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services,account_age,txn_cnt_3_credit,txn_cnt_3_debit,txn_cnt_3_services,txn_cnt_2_credit,txn_cnt_2_debit,txn_cnt_2_services,txn_amt_3_credit,txn_amt_3_debit,txn_amt_3_services,txn_amt_2_credit,txn_amt_2_debit,txn_amt_2_services,second_last,last,is_churn
0,1000000,1,7,0,128.90,1990.16,0.00,160,243,-1,648,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
1,1000001,4,3,1,2094.69,1181.73,870.82,486,338,403,688,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1000002,3,3,2,2184.48,1357.24,1282.57,147,76,310,613,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1000003,2,4,3,965.01,1861.11,1387.66,209,6,23,493,0.0,0.0,0.0,0.0,1.0,1.0,0.00,0.0,0.00,0.00,233.94,843.25,0,6,1.0
4,1000004,2,4,0,1412.47,1016.07,0.00,86,88,-1,486,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7,1,2,3202.84,631.76,1350.68,44,72,371,609,1.0,0.0,0.0,0.0,0.0,0.0,74.04,0.0,0.00,0.00,0.00,0.00,1,0,1.0
999930,1999996,4,5,2,1748.08,2244.33,978.72,29,358,219,630,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,0.00,625.19,0.00,0.00,0,1,1.0
999931,1999997,1,6,1,933.60,2117.65,778.14,410,82,172,578,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
999932,1999998,3,3,3,1124.25,1288.61,1734.86,211,121,36,454,0.0,0.0,1.0,0.0,0.0,0.0,0.00,0.0,746.76,0.00,0.00,0.00,3,0,1.0


In [16]:
all_facts = all_facts.fillna(0)
all_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services,account_age,txn_cnt_3_credit,txn_cnt_3_debit,txn_cnt_3_services,txn_cnt_2_credit,txn_cnt_2_debit,txn_cnt_2_services,txn_amt_3_credit,txn_amt_3_debit,txn_amt_3_services,txn_amt_2_credit,txn_amt_2_debit,txn_amt_2_services,second_last,last,is_churn
0,1000000,1,7,0,128.90,1990.16,0.00,160,243,-1,648,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
1,1000001,4,3,1,2094.69,1181.73,870.82,486,338,403,688,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
2,1000002,3,3,2,2184.48,1357.24,1282.57,147,76,310,613,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
3,1000003,2,4,3,965.01,1861.11,1387.66,209,6,23,493,0.0,0.0,0.0,0.0,1.0,1.0,0.00,0.0,0.00,0.00,233.94,843.25,0,6,1.0
4,1000004,2,4,0,1412.47,1016.07,0.00,86,88,-1,486,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7,1,2,3202.84,631.76,1350.68,44,72,371,609,1.0,0.0,0.0,0.0,0.0,0.0,74.04,0.0,0.00,0.00,0.00,0.00,1,0,1.0
999930,1999996,4,5,2,1748.08,2244.33,978.72,29,358,219,630,0.0,0.0,0.0,1.0,0.0,0.0,0.00,0.0,0.00,625.19,0.00,0.00,0,1,1.0
999931,1999997,1,6,1,933.60,2117.65,778.14,410,82,172,578,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0.0
999932,1999998,3,3,3,1124.25,1288.61,1734.86,211,121,36,454,0.0,0.0,1.0,0.0,0.0,0.0,0.00,0.0,746.76,0.00,0.00,0.00,3,0,1.0


In [17]:
all_facts = all_facts.astype({"txn_cnt_3_credit":'int',"txn_cnt_3_debit":'int',"txn_cnt_3_services":'int',"txn_cnt_2_credit":'int',"txn_cnt_2_debit":'int',"txn_cnt_2_services":'int',"second_last":'int',"last":'int',"is_churn":'int'})
all_facts

,account_id,overall_cnt_credit,overall_cnt_debit,overall_cnt_services,overall_amt_credit,overall_amt_debit,overall_amt_services,recency_credit,recency_debit,recency_services,account_age,txn_cnt_3_credit,txn_cnt_3_debit,txn_cnt_3_services,txn_cnt_2_credit,txn_cnt_2_debit,txn_cnt_2_services,txn_amt_3_credit,txn_amt_3_debit,txn_amt_3_services,txn_amt_2_credit,txn_amt_2_debit,txn_amt_2_services,second_last,last,is_churn
0,1000000,1,7,0,128.90,1990.16,0.00,160,243,-1,648,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0
1,1000001,4,3,1,2094.69,1181.73,870.82,486,338,403,688,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0
2,1000002,3,3,2,2184.48,1357.24,1282.57,147,76,310,613,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0
3,1000003,2,4,3,965.01,1861.11,1387.66,209,6,23,493,0,0,0,0,1,1,0.00,0.0,0.00,0.00,233.94,843.25,0,6,1
4,1000004,2,4,0,1412.47,1016.07,0.00,86,88,-1,486,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
999929,1999995,7,1,2,3202.84,631.76,1350.68,44,72,371,609,1,0,0,0,0,0,74.04,0.0,0.00,0.00,0.00,0.00,1,0,1
999930,1999996,4,5,2,1748.08,2244.33,978.72,29,358,219,630,0,0,0,1,0,0,0.00,0.0,0.00,625.19,0.00,0.00,0,1,1
999931,1999997,1,6,1,933.60,2117.65,778.14,410,82,172,578,0,0,0,0,0,0,0.00,0.0,0.00,0.00,0.00,0.00,0,0,0
999932,1999998,3,3,3,1124.25,1288.61,1734.86,211,121,36,454,0,0,1,0,0,0,0.00,0.0,746.76,0.00,0.00,0.00,3,0,1


In [18]:
all_facts.to_parquet("result/facts/all_facts.parquet")